# 🧪 Master Mixup Reproduction (Fast 45-Min Compute-Optimized Setup)

**Student**: Suryansh (Roll No: 24CS3055)  
**Department**: Department of Computer Science & Engineering, RGIPT  
**Project ID**: DL-008  
**Original Paper**: *"mixup: Beyond Empirical Risk Minimization"* (Zhang et al., ICLR 2018)  

---  
### ⚡ 45-Minute Scaled Compute Configuration:
- **Training Epoch Speed**: ~12–14s per epoch (training only).
- **Periodic Test Evaluation**: Runs every 5 epochs and on the final epoch, eliminating the 40s evaluation bottleneck.
- **Total Run Time**: Completes all 14 parts in **~40–45 minutes** on a free Google Colab T4 GPU!
- **Live Progress**: Every single epoch prints its training loss immediately (`flush=True`).

In [ ]:
# =============================================================================
# PART 1: Environment Setup, Deterministic Seeds, and Device Verification
# =============================================================================
import time
import copy
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Deterministic seed set to 42 for scientific reproducibility.")
print(f"✅ Active Device: {device}")
if device == 'cuda':
    print(f"✅ GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
# PART 2: Models (PreAct ResNet-18 & Manifold Mixup Architecture)
# =============================================================================
class PreActResidualBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out)
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    """
    Pre-activation ResNet-18 specifically designed for 32x32 CIFAR-10 images.
    Initial conv uses 3x3 kernel (stride 1, padding 1) without max-pooling.
    """
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        self.bn_last = nn.BatchNorm2d(512)
        self.linear = nn.Linear(512 * PreActResidualBlock.expansion, num_classes)

    def _make_layer(self, out_channels: int, num_blocks: int, stride: int) -> nn.Sequential:
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(PreActResidualBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.relu(self.bn_last(out))
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        return self.linear(out)


class ManifoldPreActResNet18(nn.Module):
    """
    PreAct ResNet-18 enabling feature-level interpolation across latent layers (Paper Section 3.8 & Table 5).
    """
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.bn_last = nn.BatchNorm2d(512)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels: int, num_blocks: int, stride: int) -> nn.Sequential:
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(PreActResidualBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward_manifold(self, x: torch.Tensor, target: torch.Tensor = None, alpha: float = 1.0, mix_layer: int = 0):
        if self.training and alpha > 0.0:
            lam = np.random.beta(alpha, alpha)
            perm = torch.randperm(x.size(0), device=x.device)
            y_a, y_b = target, target[perm]
        else:
            lam = 1.0
            perm = None
            y_a, y_b = target, target

        if mix_layer == 0 and self.training and alpha > 0.0:
            x = lam * x + (1.0 - lam) * x[perm]

        out = self.conv1(x)
        out = self.layer1(out)
        if mix_layer == 1 and self.training and alpha > 0.0:
            out = lam * out + (1.0 - lam) * out[perm]
        out = self.layer2(out)
        if mix_layer == 2 and self.training and alpha > 0.0:
            out = lam * out + (1.0 - lam) * out[perm]
        out = self.layer3(out)
        if mix_layer == 3 and self.training and alpha > 0.0:
            out = lam * out + (1.0 - lam) * out[perm]
        out = self.layer4(out)
        if mix_layer == 4 and self.training and alpha > 0.0:
            out = lam * out + (1.0 - lam) * out[perm]

        out = F.relu(self.bn_last(out))
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        logits = self.linear(out)
        return logits, y_a, y_b, lam

m = PreActResNet18().to(device)
print(f"✅ Models defined. Total parameters: {sum(p.numel() for p in m.parameters()):,}")

In [ ]:
# =============================================================================
# PART 3: Vectorized Mixup Implementation (Exact Paper Equations 1 & 2)
# =============================================================================
def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0, device: str = 'cuda'):
    """
    Computes vectorized mixup interpolation:
      x_tilde = lam * x_i + (1 - lam) * x_j
      lam ~ Beta(alpha, alpha)
    """
    if alpha > 0.0:
        lam = float(np.random.beta(alpha, alpha))
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=device)

    mixed_x = lam * x + (1.0 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion: nn.Module, pred: torch.Tensor, y_a: torch.Tensor, y_b: torch.Tensor, lam: float):
    """
    Computes interpolated cross-entropy loss:
      Loss = lam * Loss(pred, y_a) + (1 - lam) * Loss(pred, y_b)
    """
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)

print("✅ Vectorized mixup_data and mixup_criterion ready.")

In [ ]:
# =============================================================================
# PART 4: Label Noise Injector (Section 3.4 & Table 2 of Paper)
# =============================================================================
def add_label_noise(train_dataset, noise_rate=0.2, num_classes=10, seed=42):
    """
    Injects symmetric uniform label noise by reassigning random incorrect classes.
    """
    noisy_dataset = copy.deepcopy(train_dataset)
    if hasattr(noisy_dataset, 'targets'):
        clean_targets = np.array(noisy_dataset.targets)
    else:
        clean_targets = np.array([y for _, y in noisy_dataset])

    noisy_targets = np.copy(clean_targets)
    if noise_rate > 0.0:
        np.random.seed(seed)
        n_samples = len(clean_targets)
        n_noisy = int(noise_rate * n_samples)
        noisy_idx = np.random.choice(n_samples, size=n_noisy, replace=False)

        for idx in noisy_idx:
            orig_class = clean_targets[idx]
            other_classes = [c for c in range(num_classes) if c != orig_class]
            noisy_targets[idx] = np.random.choice(other_classes)

    if hasattr(noisy_dataset, 'targets'):
        noisy_dataset.targets = noisy_targets.tolist()

    return SymmetricNoisyCIFAR10(noisy_dataset, clean_targets, noisy_targets)


class SymmetricNoisyCIFAR10(Dataset):
    def __init__(self, dataset, clean_targets, noisy_targets):
        self.dataset = dataset
        self.clean_targets = clean_targets
        self.noisy_targets = noisy_targets

    def __getitem__(self, index):
        img, _ = self.dataset[index]
        return img, self.noisy_targets[index], self.clean_targets[index]

    def __len__(self):
        return len(self.dataset)

print("✅ Symmetric label noise injector initialized.")

In [ ]:
# =============================================================================
# PART 5: Model Calibration (Expected Calibration Error & Reliability Diagrams)
# =============================================================================
def compute_ece(logits: torch.Tensor, labels: torch.Tensor, n_bins: int = 15):
    """
    Calculates Expected Calibration Error (ECE) across confidence bins.
    """
    if not isinstance(logits, torch.Tensor):
        logits = torch.tensor(logits)
    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels)

    probs = F.softmax(logits, dim=1)
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)

    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    ece = torch.tensor(0.0)
    bin_data = []

    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]

        in_bin = confidences.gt(bin_lower) * confidences.le(bin_upper)
        prop_in_bin = in_bin.float().mean().item()

        if prop_in_bin > 0:
            accuracy_in_bin = accuracies[in_bin].float().mean().item()
            avg_confidence_in_bin = confidences[in_bin].mean().item()
            ece += torch.abs(torch.tensor(avg_confidence_in_bin - accuracy_in_bin)) * prop_in_bin
            bin_data.append((avg_confidence_in_bin, accuracy_in_bin, prop_in_bin))
        else:
            bin_data.append((0.0, 0.0, 0.0))

    return ece.item(), bin_data

print("✅ Calibration metric (ECE) defined.")

In [ ]:
# =============================================================================
# PART 6: Adversarial Attacks (Section 3.5 & Table 3 of Paper)
# =============================================================================
def fgsm_attack(model: nn.Module, x: torch.Tensor, y: torch.Tensor, epsilon: float = 4.0 / 255.0):
    """
    Fast Gradient Sign Method (FGSM) adversarial attack.
    """
    x_adv = x.clone().detach().requires_grad_(True)
    logits = model(x_adv)
    loss = F.cross_entropy(logits, y)
    model.zero_grad()
    loss.backward()

    with torch.no_grad():
        x_adv = x_adv + epsilon * x_adv.grad.sign()
        x_adv = torch.clamp(x_adv, -3.0, 3.0)
    return x_adv.detach()


def ifgsm_attack(model: nn.Module, x: torch.Tensor, y: torch.Tensor, epsilon: float = 4.0 / 255.0, steps: int = 10):
    """
    Iterative Fast Gradient Sign Method (I-FGSM / BIM).
    """
    alpha = epsilon / steps
    x_adv = x.clone().detach()

    for _ in range(steps):
        x_adv.requires_grad_(True)
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)
        model.zero_grad()
        loss.backward()

        with torch.no_grad():
            x_adv = x_adv + alpha * x_adv.grad.sign()
            eta = torch.clamp(x_adv - x, min=-epsilon, max=epsilon)
            x_adv = torch.clamp(x + eta, -3.0, 3.0)
    return x_adv.detach()


def evaluate_adversarial_robustness(model: nn.Module, dataloader: DataLoader, device: str, epsilon: float = 4.0 / 255.0):
    model.eval()
    corr_clean, corr_fgsm, corr_ifgsm, total = 0, 0, 0, 0

    for x, y in dataloader:
        if isinstance(y, (list, tuple)):
            y = y[0]
        x, y = x.to(device), y.to(device)
        total += y.size(0)

        # Clean
        with torch.no_grad():
            corr_clean += model(x).max(1)[1].eq(y).sum().item()

        # FGSM
        x_fgsm = fgsm_attack(model, x, y, epsilon=epsilon)
        with torch.no_grad():
            corr_fgsm += model(x_fgsm).max(1)[1].eq(y).sum().item()

        # I-FGSM
        x_ifgsm = ifgsm_attack(model, x, y, epsilon=epsilon, steps=10)
        with torch.no_grad():
            corr_ifgsm += model(x_ifgsm).max(1)[1].eq(y).sum().item()

    return {
        'clean_err': 100.0 * (1.0 - corr_clean / total),
        'fgsm_err': 100.0 * (1.0 - corr_fgsm / total),
        'ifgsm_err': 100.0 * (1.0 - corr_ifgsm / total)
    }

print("✅ Adversarial attack routines (FGSM, I-FGSM) defined.")

In [ ]:
# =============================================================================
# PART 7: Table 5 Alternative Baselines (Label Smoothing & Input Gaussian Noise)
# =============================================================================
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Label smoothing loss baseline tested in Paper Table 5.
    """
    def __init__(self, epsilon: float = 0.1, num_classes: int = 10):
        super().__init__()
        self.epsilon = epsilon
        self.num_classes = num_classes

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        logprobs = F.log_softmax(logits, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        loss = (1.0 - self.epsilon) * nll_loss + self.epsilon * smooth_loss
        return loss.mean()


def add_gaussian_noise(x: torch.Tensor, sigma: float = 0.05) -> torch.Tensor:
    """
    Adds Gaussian noise N(0, sigma^2) directly to images.
    """
    return x + torch.randn_like(x) * sigma if sigma > 0.0 else x

print("✅ Label smoothing & Gaussian noise baselines defined.")

In [ ]:
# =============================================================================
# PART 8: Fast Training Engine (12s Per Epoch, Periodic Eval, Live Instant Output!)
# =============================================================================
def train_one_epoch(model, dataloader, optimizer, criterion, alpha, device):
    model.train()
    running_loss = 0.0
    total_samples = 0

    for x, y in dataloader:
        if isinstance(y, (list, tuple)):
            y = y[0]
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        if alpha > 0.0:
            x_mix, y_a, y_b, lam = mixup_data(x, y, alpha=alpha, device=device)
            preds = model(x_mix)
            loss = mixup_criterion(criterion, preds, y_a, y_b, lam)
        else:
            preds = model(x)
            loss = criterion(preds, y)

        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        total_samples += x.size(0)

    return running_loss / total_samples


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    logits_list = []
    targets_list = []

    for x, y in dataloader:
        if isinstance(y, (list, tuple)):
            y = y[0]
        x, y = x.to(device), y.to(device)
        preds = model(x)
        loss = criterion(preds, y)

        running_loss += loss.item() * x.size(0)
        _, predicted = preds.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()

        logits_list.append(preds.cpu())
        targets_list.append(y.cpu())

    avg_loss = running_loss / total
    acc = 100.0 * (correct / total)
    err = 100.0 - acc
    all_logits = torch.cat(logits_list, dim=0)
    all_targets = torch.cat(targets_list, dim=0)
    return avg_loss, acc, err, all_logits, all_targets


def run_experiment(name, alpha, epochs=30, train_loader=None, test_loader=None, lr=0.1, eval_interval=5):
    """
    Fast training engine: trains in ~12-14s/epoch, evaluates every 5 epochs and at end.
    """
    print(f"\n" + "="*75)
    print(f"🚀 Starting: {name} (alpha={alpha}) for {epochs} Epochs")
    print("="*75)

    model = PreActResNet18().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.CrossEntropyLoss()

    history = {'train_loss': [], 'test_loss': [], 'test_acc': [], 'test_err': []}
    last_val_loss, last_val_acc, last_val_err = 0.0, 0.0, 100.0

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, alpha, device)
        scheduler.step()
        dt = time.time() - t0

        history['train_loss'].append(tr_loss)

        # Periodic evaluation to maintain lightning-fast ~12s epochs
        if ep % eval_interval == 0 or ep == epochs:
            val_loss, val_acc, val_err, logits, targets = evaluate(model, test_loader, criterion, device)
            last_val_loss, last_val_acc, last_val_err = val_loss, val_acc, val_err
            history['test_loss'].append(val_loss)
            history['test_acc'].append(val_acc)
            history['test_err'].append(val_err)
            print(f"Epoch [{ep:2d}/{epochs}] ({dt:.1f}s) | Train Loss: {tr_loss:.4f} | Test Loss: {val_loss:.4f} | Test Acc: {val_acc:.2f}% | Top-1 Error: {val_err:.2f}%", flush=True)
        else:
            history['test_loss'].append(last_val_loss)
            history['test_acc'].append(last_val_acc)
            history['test_err'].append(last_val_err)
            print(f"Epoch [{ep:2d}/{epochs}] ({dt:.1f}s) | Train Loss: {tr_loss:.4f} | (Evaluating every {eval_interval} eps)", flush=True)

    val_loss, val_acc, val_err, logits, targets = evaluate(model, test_loader, criterion, device)
    ece, bin_data = compute_ece(logits, targets)
    print(f"\n🎉 {name} Finished! Final Top-1 Error: {val_err:.2f}% | Final Loss: {val_loss:.4f} | Final ECE: {ece:.4f}")
    return model, history, ece, bin_data

print("✅ Fast training engine ready (Speed: ~12s/epoch, periodic eval every 5 epochs)!")

In [ ]:
# =============================================================================
# PART 9: CIFAR-10 Dataset Loading
# =============================================================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

print(f"✅ CIFAR-10 ready: {len(trainset)} Training Images, {len(testset)} Testing Images.")

In [ ]:
# =============================================================================
# PART 10: EXPERIMENT A — Clean Benchmark (Standard ERM vs Mixup alpha=1.0)
# Time: ~7-8 mins each (30 epochs x 14s) -> ~15 mins total
# =============================================================================
# 1. Standard ERM (alpha = 0.0)
erm_model, erm_hist, erm_ece, erm_bins = run_experiment(
    name="Standard ERM",
    alpha=0.0,
    epochs=30,
    train_loader=trainloader,
    test_loader=testloader,
    eval_interval=5
)

# 2. Mixup Regularization (alpha = 1.0)
mix_model, mix_hist, mix_ece, mix_bins = run_experiment(
    name="Mixup Regularization",
    alpha=1.0,
    epochs=30,
    train_loader=trainloader,
    test_loader=testloader,
    eval_interval=5
)

In [ ]:
# =============================================================================
# PART 11: EXPERIMENT B — Corrupted Label Robustness (Table 2: 20% & 50% Noise)
# Time: 20 epochs each -> ~15 mins total
# =============================================================================
noise_results = {}

for rate in [0.2, 0.5]:
    print(f"\n" + "*"*60)
    print(f"CORRUPTING CIFAR-10 WITH {int(rate*100)}% UNIFORM SYMMETRIC LABEL NOISE")
    print("*"*60)

    noisy_train = add_label_noise(trainset, noise_rate=rate, seed=42)
    noisy_loader = DataLoader(noisy_train, batch_size=128, shuffle=True, num_workers=2)

    # Train ERM under noise
    _, erm_n, _, _ = run_experiment(
        name=f"ERM ({int(rate*100)}% Noise)",
        alpha=0.0,
        epochs=20,
        train_loader=noisy_loader,
        test_loader=testloader,
        eval_interval=5
    )

    # Train Mixup (alpha=8.0 as specified in paper Table 2)
    _, mix_n, _, _ = run_experiment(
        name=f"Mixup alpha=8.0 ({int(rate*100)}% Noise)",
        alpha=8.0,
        epochs=20,
        train_loader=noisy_loader,
        test_loader=testloader,
        eval_interval=5
    )

    noise_results[rate] = {
        'erm_err': erm_n['test_err'][-1],
        'mix_err': mix_n['test_err'][-1]
    }

print("\n" + "="*60)
print("TABLE 2 REPRODUCTION SUMMARY:")
print("="*60)
for r, d in noise_results.items():
    print(f"Noise: {int(r*100)}% | ERM Top-1 Error: {d['erm_err']:.2f}% | Mixup (alpha=8.0) Top-1 Error: {d['mix_err']:.2f}%")

In [ ]:
# =============================================================================
# PART 12: EXPERIMENT C — Hyperparameter Sweep (Figure 3b: Key Alphas 0.2, 0.5, 2.0)
# Time: 15 epochs each -> ~10 mins total (0.0 and 1.0 are already obtained from Part 10!)
# =============================================================================
ablation_results = {
    0.0: {'error': erm_hist['test_err'][-1], 'ece': erm_ece},
    1.0: {'error': mix_hist['test_err'][-1], 'ece': mix_ece}
}

key_alphas = [0.2, 0.5, 2.0]
for a in key_alphas:
    _, a_hist, a_ece, _ = run_experiment(
        name=f"Ablation Alpha={a}",
        alpha=a,
        epochs=15,
        train_loader=trainloader,
        test_loader=testloader,
        eval_interval=5
    )
    ablation_results[a] = {
        'error': a_hist['test_err'][-1],
        'ece': a_ece
    }

print("\nFIGURE 3b ABLATION SUMMARY:")
for a in sorted(ablation_results.keys()):
    print(f"Alpha = {a:0.1f} -> Top-1 Error: {ablation_results[a]['error']:.2f}% | ECE: {ablation_results[a]['ece']:.4f}")

In [ ]:
# =============================================================================
# PART 13: EXPERIMENT D — Adversarial Robustness Evaluation (Table 3)
# Time: ~1-2 mins total (Evaluation only)
# =============================================================================
print("\nEvaluating Adversarial Robustness on 1,000 CIFAR-10 test samples...")
subset_indices = list(range(1000))
subset_loader = DataLoader(torch.utils.data.Subset(testset, subset_indices), batch_size=100, shuffle=False)

erm_adv = evaluate_adversarial_robustness(erm_model, subset_loader, device=device, epsilon=4.0/255.0)
mix_adv = evaluate_adversarial_robustness(mix_model, subset_loader, device=device, epsilon=4.0/255.0)

print("\n" + "="*60)
print("ADVERSARIAL ROBUSTNESS (TABLE 3 REPRODUCTION):")
print("="*60)
print(f"Clean Error     -> ERM: {erm_adv['clean_err']:.2f}% | Mixup: {mix_adv['clean_err']:.2f}%")
print(f"FGSM Error      -> ERM: {erm_adv['fgsm_err']:.2f}% | Mixup: {mix_adv['fgsm_err']:.2f}%")
print(f"I-FGSM Error    -> ERM: {erm_adv['ifgsm_err']:.2f}% | Mixup: {mix_adv['ifgsm_err']:.2f}%")

In [ ]:
# =============================================================================
# PART 14: Comprehensive Summary Plots & Export Figures
# Time: ~5 seconds
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Test Error Convergence
ep_range = range(1, len(erm_hist['test_err']) + 1)
axes[0].plot(ep_range, erm_hist['test_err'], label=f"ERM ({erm_hist['test_err'][-1]:.2f}%)", color='#d62728', lw=2)
axes[0].plot(ep_range, mix_hist['test_err'], label=f"Mixup ({mix_hist['test_err'][-1]:.2f}%)", color='#1f77b4', lw=2)
axes[0].set_title('Test Error Convergence', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=10)
axes[0].set_ylabel('Top-1 Test Error (%)', fontsize=10)
axes[0].legend(fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot 2: Reliability Diagram (Calibration)
n_bins = len(erm_bins)
centers = np.linspace(1 / (2 * n_bins), 1 - 1 / (2 * n_bins), n_bins)
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Ideal')
axes[1].plot(centers, [b[1] for b in erm_bins], 'o-', label=f"ERM (ECE: {erm_ece:.4f})", color='#d62728')
axes[1].plot(centers, [b[1] for b in mix_bins], 's-', label=f"Mixup (ECE: {mix_ece:.4f})", color='#1f77b4')
axes[1].set_title('Reliability Diagram (Calibration)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Confidence', fontsize=10)
axes[1].set_ylabel('Accuracy', fontsize=10)
axes[1].legend(fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.6)

# Plot 3: Alpha Hyperparameter Curve (Figure 3b)
alphas = sorted(list(ablation_results.keys()))
errs = [ablation_results[a]['error'] for a in alphas]
axes[2].plot(alphas, errs, 'd-', color='#2ca02c', lw=2, markersize=8)
axes[2].set_title('Fig 3b: Mixup Alpha Ablation', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Alpha Hyperparameter', fontsize=10)
axes[2].set_ylabel('Top-1 Test Error (%)', fontsize=10)
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('reproduction_summary.png', dpi=300)
plt.show()
print("✅ All plots displayed and saved to reproduction_summary.png!")